### Documentation Technique : Pipeline CAPITALIS V2

Ce pipeline automatise la sélection d'actifs et l'optimisation de portefeuille via une approche quantitative systématique.

#### 1. Phase de Screener (Filtrage de Marché)

Le module `main_screener.py` effectue un balayage des indices majeurs pour isoler les entreprises les plus prometteuses sur la base de critères fondamentaux stricts (définis dans `settings.py`) :

* **Efficacité du Capital** : `returnOnEquity >= 15%`
* **Croissance** : `earningsGrowth >= 5%`
* **Solvabilité** : `currentRatio >= 1.2`
* **Valorisation** : `forwardPE <= 30`

#### 2. Phase de Scorer (Évaluation Quantitative)

Le module `equity_scorer_F_score.py` attribue un score composite aux actions filtrées. Ce score est une synthèse pondérée de plusieurs facteurs :

* **Qualité (35%)** : F-Score (Santé financière) et ROE.
* **Valeur (25%)** : Sous-évaluation (Modèle de Graham) et PE Ratio.
* **Momentum/Risque (40%)** : Ratio de Sharpe, CAGR, Croissance des bénéfices et Volatilité.

**Formule du Score Global :**
$$\begin{aligned}
Score = & \quad (Z_{F\_Score} \times w_1) + (Z_{returnOnEquity} \times w_2) \\
& + (Z_{pct\_undervaluation} \times w_3) + (Z_{trailingPE} \times w_4) \\
& + (Z_{Sharpe} \times w_5) + (Z_{CAGR} \times w_6) \\
& + (Z_{earningsGrowth} \times w_7) + (Z_{Vol\_Ann} \times w_8)
\end{aligned}$$

- 📈 Les vecteurs de croissance (Poids Positifs) : Pour des métriques comme le ROE ou le CAGR, plus c'est haut, mieux c'est. Un Z-score positif fort est multiplié par un poids positif ($w > 0$) pour récompenser l'entreprise et booster son score final.
- 📉 Les vecteurs de friction (Poids Négatifs) : Pour des métriques comme le ratio PE (valorisation) ou la Volatilité (risque), plus c'est haut, pire c'est. Une action hors de prix aura un Z-score positif fort. En le multipliant par un poids négatif ($w < 0$), la formule pénalise mathématiquement l'action.
- 💡 Le bonus "Value" : À l'inverse, une entreprise sous-évaluée aura un PE bas (Z-score négatif). Multiplié par un poids négatif, cela crée un bonus positif ($- \times - = +$). L'algorithme prime mathématiquement les bonnes affaires !

#### 3. Optimisation HRP (Hierarchical Risk Parity)

L'allocation finale est gérée par le moteur `hrp_engine.py`, basé sur l'algorithme de Lopez de Prado. Contrairement à l'optimisation de Markowitz qui est très sensible aux erreurs d'estimation de la matrice de covariance (et qui nécessite son inversion), HRP est plus robuste car il utilise une approche hiérarchique pour diversifier le risque.

L'algorithme fonctionne en trois étapes clés :

1.  **Clustering Hiérarchique (Tree Clustering)**
    *   D'abord, on calcule la matrice de corrélation `ρ` des rendements des actifs.
    *   On la transforme en une matrice de distance `D` qui mesure la "similarité" entre les actifs. La distance entre deux actifs `i` et `j` est donnée par :
        $$ D_{i,j} = \sqrt{0.5 \times (1 - \rho_{i,j})} $$
    *   Un algorithme de clustering hiérarchique est ensuite utilisé pour regrouper les actifs en fonction de leur distance, créant une structure arborescente (un dendrogramme) qui représente la hiérarchie des risques.

2.  **Sériation de la Matrice (Quasi-Diagonalization)**
    *   Les lignes et colonnes de la matrice de covariance sont réorganisées en suivant l'arbre de clustering.
    *   Cette étape place les actifs les plus similaires les uns à côté des autres, formant des blocs de forte covariance le long de la diagonale. Cela révèle visuellement la structure hiérarchique du risque du portefeuille.

3.  **Allocation Récursive (Recursive Bisection)**
    *   L'algorithme parcourt l'arbre de haut en bas. À chaque nœud, il divise le cluster en deux sous-clusters.
    *   Il calcule la variance totale de chaque sous-cluster (en utilisant la matrice de covariance réorganisée).
    *   Le poids total alloué au cluster parent est réparti entre les deux sous-clusters en utilisant une logique de parité de risque inverse. Le sous-cluster le moins risqué reçoit une plus grande part du poids. La proportion `α` allouée au premier sous-cluster (de variance `V_1`) par rapport au second (de variance `V_2`) est :
        $$ \alpha = 1 - \frac{V_1}{V_1 + V_2} $$
    *   Ce processus est répété récursivement jusqu'à ce que les poids de chaque actif individuel soient déterminés. L'allocation finale respecte ainsi la structure de corrélation naturelle des actifs.



In [2]:
import sys
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().parent.parent
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

import settings
from src.screener.main_screener import Screener
from src.scorer.scorer import Scorer
from portfolio_optimizer import Portfolio
#from engine_constraints import run_sector_constrained_optimization

ModuleNotFoundError: No module named 'settings'

### Rappel


* **Efficacité du Capital** : `returnOnEquity >= 15%`

* **Croissance** : `earningsGrowth >= 5%`

* **Solvabilité** : `currentRatio >= 1.2`

* **Valorisation** : `forwardPE <= 27`

In [ ]:
screener = Screener(settings.RULES_DICT, settings.LINKS)
#all_tickers = screener.MarketRecupTicker()
#df_screened = screener.GetScreened(all_tickers)

Pour la partie scorer, la subtilité ici, reside dans le fait que nous prenions les outliers du "Total Score". En l'occurence ici nous prenons les 20% afin d'isoler le top 20% de notre univers d'investissement

De plus les scores sont calculer par rapport aux entreprises et leur secteur d'activité. En effet une banque et une entrerpise tech n'ont pas les mêmes referentiel de PE ratio par exemple meme si les deux peuvent être considéré comme sous evaluées.

In [ ]:
scorer = Scorer() 
scored_df = scorer.ComputeScore()

seuil_top_20 = scored_df['Total_Score'].quantile(0.85)

outliers_df = scored_df[scored_df['Total_Score'] >= seuil_top_20].copy()

outliers_df.to_csv(settings.FINAL_SCORED_PATH, index=False)


2026-03-21 22:11:16 - INFO - scorer - Lecture des actifs depuis /Users/leolombardini/Documents/CAPITALIS V2/src/data/screened_assets.csv
2026-03-21 22:11:16 - INFO - scorer - Taux sans risque (TNX 10 ans) : 0.0439
2026-03-21 22:11:19 - INFO - scorer - Récupération des ratios fondamentaux manquants pour 94 actifs (via multithreading)...
2026-03-21 22:11:21 - INFO - scorer - Scoring sectoriel activé sur la colonne : sector


In [ ]:
outliers_df = outliers_df.head(10)
auto_tickers = outliers_df['Ticker'].tolist()


MANUAL_LIST = ["BE","CAT","VOLV-B.ST", "COHR","GS","IONQ"]

final_portfolio_tickers = list(set(auto_tickers + MANUAL_LIST))

pf = Portfolio(final_portfolio_tickers, max_weight=0.1725, min_weight=0.02, capital=18000,risk_aversion=3)

pf.Optimize(method="HRP")
pf.DisplayResults()

NameError: name 'outliers_df' is not defined

### 4. Optimisation Mean-Variance (Markowitz)

En complément du HRP, nous pouvons utiliser l'optimisation Mean-Variance de Markowitz. Cette méthode cherche à maximiser le rendement attendu pour un niveau de risque donné (ou minimiser le risque pour un rendement donné).

L'optimisation est basée sur la fonction d'utilité suivante:

$$ U(w) = w^T \mu - rac{\lambda}{2} w^T \Sigma w $$

Où:
- $ w $ sont les poids du portefeuille
- $\mu$ sont les rendements attendus
- $\Sigma$ est la matrice de covariance
- $\lambda$ est le coefficient d'aversion au risque.

Un $\lambda$ élevé pénalise la variance et conduit à un portefeuille plus prudent. Un $\lambda$ faible favorise le rendement.

In [ ]:
# Optimisation Mean-Variance
pf_mean_var = Portfolio(final_portfolio_tickers, max_weight=0.20, min_weight=0.0, capital=20000, risk_aversion=3)
weights_mean_var = pf_mean_var.Optimize(method='MeanVar')
pf_mean_var.DisplayResults()

2026-03-21 22:11:25 - INFO - portfolio_optimizer - Initialisation du portefeuille avec 16 tickers.
2026-03-21 22:11:25 - INFO - portfolio_optimizer - Lancement de l'optimisation avec la méthode : MEANVAR
2026-03-21 22:11:25 - INFO - portfolio_optimizer - Contraintes de poids : Min=0.00%, Max=20.00%
2026-03-21 22:11:25 - INFO - portfolio_optimizer - Optimisation terminée.
2026-03-21 22:11:25 - INFO - portfolio_optimizer - Récupération des métadonnées pour les actifs manuels...


Lecture des rendements depuis le cache : /Users/leolombardini/Documents/CAPITALIS V2/src/data/cache/returns_BE_CAT_COHR_CRH_DECK_EIX_ELV_EME_GS_HIG_IBE.MC_IONQ_MTX.DE_MU_NVDA_VOLV-B.ST_10y_1d.csv


2026-03-21 22:11:26 - INFO - portfolio_optimizer - Génération des graphiques d'allocation...


2026-03-21 22:11:26 - INFO - portfolio_optimizer - Affichage des résultats terminé.



··························································································
  TICKER     | NOM                       | SECTEUR              | POIDS      | MONTANT
··························································································
  BE         | Bloom Energy Corporatio   | Industrials          |    20.00% |  4000.00€
  COHR       | Coherent Corp.            | Technology           |    20.00% |  4000.00€
  CAT        | Caterpillar, Inc.         | Industrials          |    20.00% |  4000.00€
  MU         | Micron Technology, Inc.   | Technology           |    20.00% |  4000.00€
  EIX        | Edison International      | Utilities            |     9.91% |  1982.74€
  IBE.MC     | ACCIONES IBERDROLA        | Utilities            |     7.72% |  1544.64€
  EME        | EMCOR Group, Inc.         | Industrials          |     2.36% |   472.62€
  HIG        | The Hartford Insurance    | Financial Services   |     0.00% |     0.00€
  MTX.DE     | MTU Aero En

In [ ]:
#Optmisation Markowitz
pf.Optimize(method="MinVol")
pf.DisplayResults()

2026-03-21 22:11:26 - INFO - portfolio_optimizer - Lancement de l'optimisation avec la méthode : MINVOL
2026-03-21 22:11:26 - INFO - portfolio_optimizer - Contraintes de poids : Min=2.00%, Max=17.25%
2026-03-21 22:11:26 - INFO - portfolio_optimizer - Optimisation terminée.
2026-03-21 22:11:26 - INFO - portfolio_optimizer - Récupération des métadonnées pour les actifs manuels...
2026-03-21 22:11:27 - INFO - portfolio_optimizer - Génération des graphiques d'allocation...


2026-03-21 22:11:27 - INFO - portfolio_optimizer - Affichage des résultats terminé.



··························································································
  TICKER     | NOM                       | SECTEUR              | POIDS      | MONTANT
··························································································
  IBE.MC     | ACCIONES IBERDROLA        | Utilities            |    17.25% |  3105.00€
  ELV        | Elevance Health, Inc.     | Healthcare           |    15.63% |  2813.54€
  EIX        | Edison International      | Utilities            |    14.19% |  2553.56€
  VOLV-B.ST  | Volvo, AB ser. B          | Industrials          |    13.62% |  2451.37€
  HIG        | The Hartford Insurance    | Financial Services   |    10.99% |  1978.16€
  MTX.DE     | MTU Aero Engines AG       | Industrials          |     8.32% |  1498.37€
  IONQ       | IonQ, Inc.                | Technology           |     2.00% |   360.00€
  CAT        | Caterpillar, Inc.         | Industrials          |     2.00% |   360.00€
  MU         | Micron Tech

In [ ]:
#Equi Pondération
pf.Optimize(method="EqualWeights")
pf.DisplayResults()

2026-03-21 22:11:27 - INFO - portfolio_optimizer - Lancement de l'optimisation avec la méthode : EQUALWEIGHTS
2026-03-21 22:11:27 - INFO - portfolio_optimizer - Contraintes de poids : Min=2.00%, Max=17.25%
2026-03-21 22:11:27 - WARNING - portfolio_optimizer - Les contraintes de poids ne sont pas applicables à la méthode 'EqualWeights'.
2026-03-21 22:11:27 - INFO - portfolio_optimizer - Optimisation terminée.
2026-03-21 22:11:27 - INFO - portfolio_optimizer - Récupération des métadonnées pour les actifs manuels...
2026-03-21 22:11:27 - INFO - portfolio_optimizer - Génération des graphiques d'allocation...


2026-03-21 22:11:27 - INFO - portfolio_optimizer - Affichage des résultats terminé.



··························································································
  TICKER     | NOM                       | SECTEUR              | POIDS      | MONTANT
··························································································
  BE         | Bloom Energy Corporatio   | Industrials          |     6.25% |  1125.00€
  GS         | Goldman Sachs Group, In   | Financial Services   |     6.25% |  1125.00€
  MU         | Micron Technology, Inc.   | Technology           |     6.25% |  1125.00€
  MTX.DE     | MTU Aero Engines AG       | Industrials          |     6.25% |  1125.00€
  VOLV-B.ST  | Volvo, AB ser. B          | Industrials          |     6.25% |  1125.00€
  DECK       | Deckers Outdoor Corpora   | Consumer Cyclical    |     6.25% |  1125.00€
  COHR       | Coherent Corp.            | Technology           |     6.25% |  1125.00€
  CAT        | Caterpillar, Inc.         | Industrials          |     6.25% |  1125.00€
  IBE.MC     | ACCIONES IB

### 5. Stratégie d'Allocation Avancée : Contraintes Sectorielles & Régimes de Marché

Cette étape du pipeline introduit une intelligence hybride, mêlant convictions stratégiques et modélisation statistique avancée.

#### A. Optimisation Sectorielle Contrainte (`engine_constraints.py`)

L'approche standard HRP (Hierarchical Risk Parity) est purement mathématique et "aveugle" aux secteurs. Bien qu'efficace pour réduire le risque, elle peut parfois sous-allouer des secteurs porteurs si leur volatilité est élevée (comme la Tech).

Pour pallier cela, nous utilisons une approche **Top-Down** :

1.  **Définition Macro (Top)** : Une allocation cible par secteur est définie (ex: *Technologie 40%, Santé 20%*...). Cela permet d'injecter une conviction directionnelle forte.
2.  **Optimisation Micro (Down)** : À l'intérieur de chaque poche sectorielle, l'algorithme HRP est exécuté pour répartir le capital de manière optimale entre les actifs du secteur.

$$ Poids_{Actif} = Poids_{Secteur (Cible)} \times Poids_{HRP (Intra-Secteur)} $$

Cette méthode garantit que le portefeuille respecte la structure macro-économique souhaitée tout en minimisant le risque idiosyncratique (spécifique) de chaque action au sein de son groupe.

#### B. Filtre de Régime Macro (Hidden Markov Model - HMM)

Pour déterminer "quels poids donner aux secteurs" de manière scientifique plutôt que manuelle, nous intégrons un **Modèle de Markov Caché (HMM)**.

Les marchés financiers alternent entre des "régimes" invisibles (d'où le terme "Caché") : par exemple, un *Régime Haussier à faible volatilité* versus un *Régime de Krach à haute volatilité*. Le HMM analyse les séries temporelles pour calculer la probabilité d'être dans un état donné à l'instant T.

**Fonctionnement du filtre :**

*   **Détection :** Le modèle estime le régime de marché actuel.
*   **Adaptation Dynamique :** Les pondérations sectorielles cibles (`sector_allocation_target`) changent selon le régime :
    *   *Régime "Risk-On" (Expansion)* : Allocation agressive. Sur-pondération de la Technologie, de l'Industrie et des facteurs Momentum.
    *   *Régime "Risk-Off" (Contraction)* : Allocation défensive. Rotation vers la Santé, la Consommation de Base (Staples) et les Utilities, voire augmentation du Cash.

Cela transforme le pipeline en un système adaptatif qui ajuste son exposition au risque en fonction de la "météo" du marché.


In [ ]:
CAPITAL = 10000
ALLOCATION = {'Technology': 0.40, 'Healthcare': 0.20, 'Industrials': 0.20, 'Financial Services': 0.20}
run_sector_constrained_optimization(CAPITAL, ALLOCATION)

NameError: name 'run_sector_constrained_optimization' is not defined